Это ноутбук для обучения YOLOv8 для infraPPE-monitor  
https://github.com/ferrovovan/infraPPE-monitor

# Шаг 0: Подготовка среды и установка библиотек

In [1]:
!pip install kaggle
!pip install ultralytics


# Вместо файла kaggle.json мы будем использовать переменные окружения.
import os
os.environ['KAGGLE_USERNAME'] = 'vash_username' # <-- ЗАМЕНИТЕ НА ВАШЕ ИМЯ ПОЛЬЗОВАТЕЛЯ KAGGLE!
os.environ['KAGGLE_KEY'] = input("Введите KAGGLE_KEY: ")


import ultralytics
ultralytics.checks()

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.1/112.6 GB disk)


# Шаг 1: Скачивание датасета


In [2]:

DATASET_REPO = 'shlokraval/ppe-dataset-yolov8'

repo_name = DATASET_REPO.split('/')[-1]
DATASET_ARCHIVE_NAME = f'{repo_name}.zip'
DATASET_DIR = 'dataset'


!kaggle datasets download -d {DATASET_REPO} #-O {DATASET_ARCHIVE_NAME}

# 4. Распакуйте архив в отдельную папку для удобства
!mkdir -p {DATASET_DIR}
!unzip -q {DATASET_ARCHIVE_NAME} -d {DATASET_DIR}

# Проверяем, что внутри (полезно для отладки)
print(f"Содержимое папки {DATASET_DIR}:")
!ls {DATASET_DIR}

Dataset URL: https://www.kaggle.com/datasets/shlokraval/ppe-dataset-yolov8
License(s): apache-2.0
100% 2.34G/2.35G [00:43<00:00, 46.2MB/s]
100% 2.35G/2.35G [00:43<00:00, 58.4MB/s]
Содержимое папки dataset:
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


# Шаг 2: Обучение модели


In [9]:
from ultralytics import YOLO
from torch.cuda import is_available as is_gpu
import os
from google.colab import drive

drive.mount('/content/drive')

# Путь к скачанному файлу data.yaml внутри папки dataset
CONFIG_PATH = 'dataset/data.yaml'
GDRIVE_PATH = '/content/drive/MyDrive/'

# Определяем параметры обучения
TOTAL_EPOCHS = 75
# SAVE_PERIOD = 3  # сохраняет доп. файлы каждые 3 эпох
IMG_SIZE = 640
if is_gpu:
    BATCH_SIZE = 96  # Для Tesla T4, 15095MiB
else:
    print("На CPU слишком долго. Борода.")
    quit()
    #BATCH_SIZE = -1

PROJECT_DIR = GDRIVE_PATH + 'train_ppe_v1' # Место сохранения результатов в Colab
EXPERIMENT_NAME = 'yolov8_ppe_nano'
START_WEIGHTS_NAME = 'yolov8n.pt'
START_WEIGHTS_NAME = 'last.pt'


# Инициализации весов ---
best_weights_path = os.path.join(PROJECT_DIR, EXPERIMENT_NAME, 'weights', 'last.pt')
# print(best_weights_path)

if os.path.exists(best_weights_path):
    current_weights = best_weights_path
    print(f"Обнаружены предыдущие лучшие веса: {current_weights}.\n Обучение будет продолжено с них.")
else:
    current_weights = START_WEIGHTS_NAME
    print(f"Предыдущие лучшие веса не найдены.\n Обучение начнется с предобученной модели: {current_weights}")

# Создаем директорию проекта, если ее нет
os.makedirs(PROJECT_DIR, exist_ok=True)


# 1. Загружаем модель с последними сохраненными весами
model = YOLO(current_weights)

# 2. Запускаем обучение на EPOCHS_PER_CYCLE эпох
# Каждому циклу присваиваем уникальное имя, чтобы результаты сохранялись отдельно
results = model.train(
    data=CONFIG_PATH,
    epochs=TOTAL_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_DIR,
    name=f'{EXPERIMENT_NAME}',
    save=True,             # Сохранять модель (weights/best.pt и weights/last.pt)
    resume=True           # Возобновляет обучение с того места, где оно было прервано
    #resume=False           # Для нового обучения
)

print("\n--- Обучение завершено! ---")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Предыдущие лучшие веса не найдены.
 Обучение начнется с предобученной модели: last.pt
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=96, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=75, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mode

KeyboardInterrupt: 

# Дополнительно. Чистка диска.
(Не рабочая)

In [ ]:
best_model_path = results.best
stripped_model_path = model_path.replace('.pt', '_stripped.pt')
strip_optimizer(best_model_path, stripped_model_path)
print(f"Модель успешно зачищена. Новый файл сохранен как: {stripped_model_path}")

In [ ]:
!cp {PROJECT_DIR}/{EXPERIMENT_NAME}/weights/best.pt  {PROJECT_DIR}/{EXPERIMENT_NAME}/detect_ppe_nano.pt

In [8]:
!rm -r {PROJECT_DIR}/

In [ ]:
# Для переноса (сохранения) весов из локального на диск.
import shutil
from pathlib import Path

local_dir = Path('/content/train_ppe_v1/yolov8_ppe_nano/weights')
gdrive_dir = Path('/content/drive/MyDrive/train_ppe_v1/yolov8_ppe_nano/weights')
if os.path.exists(local_dir):
    gdrive_dir.mkdir(parents=True, exist_ok=True)
    for p in local_dir.glob('*.pt'):
        shutil.copy2(p, gdrive_dir / p.name)
    print("Копирование завершено:", list(gdrive_dir.glob('*')))